In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
jobs=pd.read_csv('/content/drive/MyDrive/DATA 641/data/fake_job_postings.csv')

In [3]:
jobs['fraudulent'].value_counts()

,count
fraudulent,
0,17014
1,866


In [4]:
indices_to_drop = jobs[jobs['fraudulent'] == 0].sample(n=16000).index #fixing class imbalance
jobs = jobs.drop(indices_to_drop)

In [5]:
jobs['fraudulent'].value_counts()

,count
fraudulent,
0,1014
1,866


**Commitments made in the project plan**

1. Methods: Logistic Regression, SVM, RNN, Random Forest, BERT Transformer
2. Utilize the same or similar pre-processing technique
3. I will then train and test each model, evaluating
accuracy, precision, recall, and the macro-averaged F1 score and utilizing CodeCarbon’s ability to
estimate CO2 Emissions to log and obtain the emissions from training the data and report the calculated
CE_Rel and delta CE_rel metrics.

In [6]:
jobs.head()

,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
15,16,VP of Sales - Vault Dragon,"SG, 01, Singapore",Sales,120000-150000,Jungle Ventures is the leading Singapore based...,About Vault Dragon Vault Dragon is Dropbox for...,Key Superpowers3-5 years of high-pressure sale...,"Basic: SGD 120,000Equity negotiable for a rock...",0,1,1,Full-time,Executive,Bachelor's Degree,Facilities Services,Sales,0
17,18,Southend-on-Sea Traineeships Under NAS 16-18 Y...,"GB, SOS, Southend-on-Sea",NaN,NaN,Established on the principles that full time e...,Government funding is only available for 16-18...,16-18 year olds only due to government funding...,Career prospects.,0,1,1,NaN,NaN,NaN,NaN,NaN,0
41,42,English Teacher Abroad,"US, CA, Sacramento",NaN,NaN,We help teachers get safe &amp; secure jobs ab...,"Play with kids, get paid for it Love travel? J...",University degree required. TEFL / TESOL / CEL...,See job description,0,1,1,Contract,NaN,Bachelor's Degree,Education Management,NaN,0
64,65,SENIOR FINANCE SOFTWARE RESEARCHER AND ENGINEER,"US, ,",NaN,NaN,NaN,DUTIES: Conduct research for building technica...,REQUIREMENTS: Bachelor’s degree in Mathematics...,NaN,0,0,0,NaN,NaN,NaN,NaN,NaN,0
75,76,Senior Full-Stack Engineer - Ruby on Rails (Te...,"DE, BE, Berlin",Engineering,NaN,Babbel enables anyone to learn languages in an...,We are looking for Senior Rails Full-Stack Eng...,5 years+ experience in delivering stunning fro...,The potential to change the way of learning fo...,0,1,1,Full-time,Mid-Senior level,Master's Degree,E-Learning,Engineering,0


In [7]:
model_df = jobs[['description', 'fraudulent']]

In [8]:
model_df.head()


,description,fraudulent
15,About Vault Dragon Vault Dragon is Dropbox for...,0
17,Government funding is only available for 16-18...,0
41,"Play with kids, get paid for it Love travel? J...",0
64,DUTIES: Conduct research for building technica...,0
75,We are looking for Senior Rails Full-Stack Eng...,0


In [9]:
import nltk
import re
import html
from nltk.corpus import stopwords
nltk.download('stopwords',quiet=True)
stop_words=set(stopwords.words('english'))


def preprocess_text(text):
  text=re.sub(r"(?:http\S+|@)","",text) #arguments are pattern,replace,string.
  text=html.unescape(text) #convert XML to string. This function can handle XML entities like &amp
  tokens=text.split() #just using split()
  tokens=[token for token in tokens if token not in stop_words]
  return ' '.join(tokens)

there's a class imbalance. much more non-fraudulent ones than fraudulent. There are ~17800 observations and maybe 800 fraudulent ones. Maybe consider fixing this (but don't think it has a direct effect on workload, but maybe want things to be realistic).

**LOGISTIC REGRESSION**

In [ ]:
##CHECK LAB 6 FOR K FOLD cross validation applied to logistic regression

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,accuracy_score

In [11]:
model_df['description']=model_df['description'].astype(str)

/tmp/ipykernel_16490/3832600678.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  model_df['description']=model_df['description'].astype(str)


In [12]:
processed_descriptions=[preprocess_text(doc) for doc in model_df['description']]

In [13]:
labels=model_df['fraudulent']

In [14]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test=train_test_split(processed_descriptions,labels,test_size=0.3,random_state=42)


In [15]:
tfidf=TfidfVectorizer()
X_tfidf = tfidf.fit_transform(x_train)
X_test_tfidf = tfidf.transform(x_test)

In [16]:
reg_classifier=LogisticRegression()
reg_classifier.fit(X_tfidf,y_train)

LogisticRegression()

In [17]:
y_pred = reg_classifier.predict(X_test_tfidf) #should break to test train

In [18]:
print(classification_report(y_test, y_pred)) #ofc it did perfectly bc i didnt split it into test.train.

              precision    recall  f1-score   support

           0       0.84      0.94      0.89       301
           1       0.92      0.80      0.86       263

    accuracy                           0.88       564
   macro avg       0.88      0.87      0.87       564
weighted avg       0.88      0.88      0.87       564



**SUPPORT VECTOR MACHINE**

In [19]:
from sklearn.model_selection import cross_validate, KFold, GridSearchCV
from sklearn.base import TransformerMixin
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix, classification_report
from sklearn import metrics
from sklearn.preprocessing import StandardScaler

In [20]:
class SparsetoDense(TransformerMixin):
  def fit(self, x, y = None, **fit_params):
    return self
  def transform(self, x, y=None, **fit_params):
    return x.toarray()

In [24]:
svm_pipe=Pipeline([
    ('densify',SparsetoDense()),
    ('scale', StandardScaler()),
    ('classify',SVC())
])

kernel= ['rbf', 'linear']
C = [0.001, 0.01, 1, 10] #if its running too long will cut down on some of these combos
svm_params = {
    'classify__kernel': kernel,
    'classify__C': C
}

In [25]:
inner_cv=KFold(n_splits=3, shuffle=True, random_state=1)
outer_cv=KFold(n_splits=5, shuffle=True, random_state=1)

grid_SVC=GridSearchCV(svm_pipe, svm_params, cv=inner_cv)

In [26]:
scores=cross_validate(grid_SVC,
                     X=X_tfidf,
                     y=y_train,
                     cv=outer_cv,
                     scoring=['accuracy','f1','precision','recall'],
                     return_estimator=True)

In [27]:
print(scores['test_accuracy'])
print(scores['test_precision'])
print(scores['test_recall'])
print(scores['test_f1'])

[0.88257576 0.84790875 0.82889734 0.84790875 0.84410646]
[0.86324786 0.8203125  0.80508475 0.82113821 0.88596491]
[0.87068966 0.86065574 0.81196581 0.8487395  0.78294574]
[0.86695279 0.84       0.80851064 0.83471074 0.83127572]


In [ ]:
#SVM Cross validation ran for 24 mins

In [28]:
grid_SVC.fit(X_tfidf,y_train)
grid_SVC.best_params_

{'classify__C': 0.001, 'classify__kernel': 'linear'}

In [29]:
#bestmodel as SVC object
#then do bestmodel.fit
#then get y_pred with bestmodel
SVC_model = SVC(kernel='linear', C=0.001)
SVC_model.fit(X_tfidf, y_train)
y_pred = SVC_model.predict(X_test_tfidf)

In [31]:
#get classifciation report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.53      1.00      0.70       301
           1       0.00      0.00      0.00       263

    accuracy                           0.53       564
   macro avg       0.27      0.50      0.35       564
weighted avg       0.28      0.53      0.37       564



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [33]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test,y_pred))

[[301   0]
 [263   0]]


**RECURRENT NEURAL NET (RNN)**

**RANDOM FOREST**

**BERT Transformer**